In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine,text
import db_connect
import pathlib
import shutil
from sqlalchemy.exc import SQLAlchemyError
from datetime import datetime

In [2]:
# Set file path
user_path = pathlib.Path.home()
if (user_path / 'Central Group/PST Performance Team - เอกสาร').exists():
    filepath = user_path / 'Central Group/PST Performance Team - เอกสาร'
else:
    filepath = user_path / 'Central Group/PST Performance Team - Documents'

In [3]:
bu = 'CFR'
sheet_name = 'Summary'
table = 'missrate'
f_path = filepath / 'Shared' / 'Performance' / 'Missrate' / 'Missrate_approved'
t_path = filepath / 'Shared' / 'Performance' / 'Missrate' / 'Missrate_imported'
error_log_path = filepath / 'Shared' / 'Performance' / 'report' / 'error_log.csv'

In [4]:
excel_files = [f for f in os.listdir(f_path) if f.endswith(('.xlsx', '.xls')) and os.path.getsize(os.path.join(f_path, f)) > 0]
print(f"Found {len(excel_files)} Excel files in the directory.")

Found 2 Excel files in the directory.


In [ ]:
# PostgreSQL connection details
engine = create_engine(db_connect.db_url_pstdb)
plan_query = f"SELECT stcode, cntdate, branch FROM planall2 WHERE bu = '{bu}'"
df_plan = pd.read_sql(plan_query, con=engine)

for file in excel_files:
    file_path = f_path / file
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, usecols='A:J', skiprows=6,
                           dtype={'Group': str, 'Department': str,'Barcode': str,
                                  'Name': str,'Count': float,'Add/Edit': float,'Scan Error': float,
                                  'PST/Store': str,'Remark (Root Cause)': str,'Note': str})

        file_parts = os.path.splitext(file)[0].split('_')
        if len(file_parts) == 4:
            cols1, cols2, cols3, cols4 = file_parts
        else:
            cols1, cols2, cols3, cols4 = None, None, None, None

        df['cols1'] = cols1
        df['bu'] = cols2
        df['stcode'] = cols3
        df['cntdate'] = cols4


        df['PST/Store'] = df['PST/Store'].str.strip().str.upper()
        df['PST/Store'] = df['PST/Store'].replace(['AJIS', 'INNOVATION', 'SSD', 'PCS'], 'OUTSOURCE')

        #df['skutype'] = 'Credit'
        #df = df[df['STORE'] == cols3]
        df = df.drop(columns=['cols1'])
        df.columns = df.columns.str.lower()
        df = df.rename(columns={'remark (root cause)': 'remark root cause'})
    
        join_plan = df.merge(df_plan[['stcode', 'cntdate','branch']], on=['stcode', 'cntdate'], how='left')
        df = join_plan
        df = df[df['branch'].notnull()]
        df = df.drop(columns=['branch'])

        # Get values
        del_stcode = df['stcode'].iloc[0]
        del_cntdate = df['cntdate'].iloc[0]

        # Prepare safe delete query
        del_query = text(f"""
            DELETE FROM {table} 
            WHERE stcode = :stcode AND cntdate = :cntdate
        """)
        del_params = {"stcode": del_stcode, "cntdate": del_cntdate}

        print(f"Processing file: {file}")

    except Exception as e:
        error_message = f"Error processing file {file}: {str(e)}"
        print(error_message)
        continue

In [14]:
# Execute query with parameter binding
for file in excel_files:
    file_path = f_path / file
    try:
        df_piece = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            usecols='L:M',
            skiprows=2,   # เริ่มอ่านตั้งแต่แถว 3 (header แถว "PST / AJIS")
            nrows=3,      # อ่าน 3 แถว: All Scan, Qty weight, SKU weight
            header=None
        )
        print(df_piece.head())
        
        print(f"Extracted summary data from columns L:M for file: {file}")
        
        # ✅ ตรวจสอบให้ครบ 3 แถว 2 คอลัมน์
        if df_piece.shape != (3, 2):
            print(f"❌ Skipping file {file} because summary data (columns L:M) is incomplete.")

        # ✅ Map ค่าเข้า dataframe
        df['all scan-pst'] = df_piece.iloc[0, 0] if df_piece.iloc[0, 0] is not None else 0
        df['qty weight-pst'] = df_piece.iloc[1, 0] if df_piece.iloc[1, 0] is not None else 0
        df['sku weright-pst'] = df_piece.iloc[2, 0] if df_piece.iloc[2, 0] is not None else 0
        
        df['all scan-outsource'] = df_piece.iloc[0, 1] if df_piece.iloc[0, 1] is not None else 0
        df['qty weight-outsource'] = df_piece.iloc[1, 1] if df_piece.iloc[1, 1] is not None else 0
        df['sku weright-outsource'] = df_piece.iloc[2, 1] if df_piece.iloc[2, 1] is not None else 0


        # Remove duplicate rows based on all columns except index
        df = df.drop_duplicates()
        # Check if 'pst/store' or 'remark root cause' have NA values; if so, skip this file
        if df['pst/store'].isna().any() or df['remark root cause'].isna().any():
            print(f"❌ Skipping file {file} due to NA in 'pst/store' or 'remark root cause'")
            
        timestamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"✅ Data inserted into '{table}' at {timestamp}")
        print(f"✅ moved file: {file} → {t_path}")
        
        
    except Exception as e:
        print(f"Error processing {file}: {e}")


           11  12
0  298642.482 NaN
1   15992.482 NaN
2     580.000 NaN
Extracted summary data from columns L:M for file: Missrate_CFR_107_20260221.xlsx
Error processing Missrate_CFR_107_20260221.xlsx: name 'df' is not defined
          11  12
0  269248.28 NaN
1   12415.28 NaN
2     705.00 NaN
Extracted summary data from columns L:M for file: Missrate_CFR_443_20260222.xlsx
Error processing Missrate_CFR_443_20260222.xlsx: name 'df' is not defined
